# 🔬 Fake News Detection using BART-Large-MNLI (Zero-Shot & Fine-Tuned)

This notebook uses **[facebook/bart-large-mnli](https://huggingface.co/facebook/bart-large-mnli)** for fake news classification using:

1. **Zero-Shot Classification** — No training needed, uses NLI-based approach
2. **Transfer Learning (Fine-Tuning)** — Train on labeled data for better accuracy

### Model Info
- **Base**: `facebook/bart-large` (406M parameters)
- **Trained on**: MultiNLI (MNLI) dataset
- **Approach**: Natural Language Inference — classifies by checking if text **entails** a hypothesis like *"This news is fake"*
- **Labels**: 0 = Contradiction, 1 = Neutral, 2 = Entailment

---
## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch pandas matplotlib seaborn

---
## 2. Imports & Device Setup

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    pipeline
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 3. 🔍 Dataset Quality Check

**IMPORTANT**: Before training any model, we must verify the dataset contains **real, meaningful text** — not synthetic/randomly generated content.

In [ ]:
# ============================================================
# UPDATE THIS PATH to your CSV file
# ============================================================
DATASET_PATH = 'fake_news_dataset.csv'

df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\n{'='*60}")
print(f"SAMPLE ARTICLES")
print(f"{'='*60}")

for label_val in df['label'].unique()[:2]:
    sample = df[df['label'] == label_val].iloc[0]
    print(f"\n--- [{label_val.upper()}] ---")
    print(f"Title: {str(sample.get('title', 'N/A'))[:100]}")
    print(f"Text:  {str(sample.get('text', 'N/A'))[:300]}")

In [ ]:
# ============================================================
# QUALITY CHECK: Detect if dataset is synthetic/random text
# Real news articles have proper grammar, sentences, and structure
# Synthetic text is just random words strung together
# ============================================================
import re

def check_text_quality(text, min_sentence_ratio=0.3):
    """Check if text looks like real prose or random word salad."""
    text = str(text)
    words = text.split()
    if len(words) < 10:
        return False, "Too short"
    
    # Check for sentence-ending punctuation
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    
    # Real text has sentences; random words don't
    has_punctuation = bool(re.search(r'[.!?,;:]', text))
    
    # Check for common English function words in proper order
    function_words = ['the', 'a', 'an', 'is', 'was', 'are', 'were', 'has', 'have', 'had',
                      'that', 'which', 'who', 'said', 'according']
    fw_count = sum(1 for w in words if w.lower() in function_words)
    fw_ratio = fw_count / len(words)
    
    # Real English text typically has 15-30% function words
    is_real = has_punctuation and fw_ratio > 0.05 and len(sentences) >= 2
    
    return is_real, f"punct={has_punctuation}, fw_ratio={fw_ratio:.2f}, sentences={len(sentences)}"

# Check a sample of articles
sample = df.sample(min(100, len(df)), random_state=42)
quality_results = sample['text'].apply(lambda x: check_text_quality(x)[0])
real_text_pct = quality_results.mean() * 100

print(f"{'='*60}")
print(f"📊 DATASET QUALITY ASSESSMENT")
print(f"{'='*60}")
print(f"Articles that look like REAL text: {real_text_pct:.1f}%")
print(f"Articles that look like RANDOM/SYNTHETIC: {100-real_text_pct:.1f}%")

if real_text_pct < 50:
    print(f"\n🚨 WARNING: This dataset appears to contain SYNTHETIC/RANDOM text!")
    print(f"   The text is random words without grammar or meaning.")
    print(f"   No ML model can learn meaningful patterns from random data.")
    print(f"   → You need a dataset with REAL news articles.")
    print(f"   → We will download and use a proper dataset below.")
    USE_REPLACEMENT_DATASET = True
else:
    print(f"\n✅ Dataset appears to contain real, meaningful text.")
    USE_REPLACEMENT_DATASET = False

---
## 4. Load a Proper Dataset (if needed)

If the original dataset was flagged as synthetic, we'll download a real fake news dataset.

**Options:**
- `GonzaloA/fake_news` — ~20K real articles (Fake/Real)
- Set `USE_REPLACEMENT_DATASET = False` above to skip this and use your own CSV

In [ ]:
if USE_REPLACEMENT_DATASET:
    print("📥 Downloading a real fake news dataset...")
    from datasets import load_dataset
    
    # Load GonzaloA/fake_news dataset from HuggingFace
    hf_dataset = load_dataset('GonzaloA/fake_news', split='train')
    df = hf_dataset.to_pandas()
    
    # Standardize columns
    # This dataset has: title, text, label (0=fake, 1=real)
    print(f"Downloaded dataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Map numeric labels to text labels
    if df['label'].dtype in ['int64', 'float64']:
        df['label'] = df['label'].map({0: 'fake', 1: 'real'})
    
    print(f"Label distribution:\n{df['label'].value_counts()}")
    
    # Show sample to verify it's real text
    print(f"\n--- SAMPLE ARTICLE ---")
    sample = df.iloc[0]
    print(f"Title: {str(sample.get('title', 'N/A'))[:100]}")
    print(f"Text:  {str(sample.get('text', 'N/A'))[:300]}")
    print(f"Label: {sample['label']}")
else:
    print("✅ Using your original dataset (passed quality check).")

---
## 5. Data Preprocessing

In [ ]:
# Clean and prepare data
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 20]  # Remove very short texts

# Combine title + text for richer context
if 'title' in df.columns:
    df['input_text'] = df['title'].fillna('').astype(str) + ' </s> ' + df['text'].astype(str)
else:
    df['input_text'] = df['text'].astype(str)

# Map labels: fake=0, real=1
label_map = {'fake': 0, 'real': 1}
df['label_id'] = df['label'].str.lower().str.strip().map(label_map)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

print(f"Final dataset size: {len(df)}")
print(f"Label distribution:\n{df['label_id'].value_counts().rename({0: 'Fake', 1: 'Real'})}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#2ecc71']
df['label_id'].value_counts().rename({0: 'Fake', 1: 'Real'}).plot(
    kind='bar', ax=axes[0], color=colors
)
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['text_length'] = df['text'].astype(str).apply(len)
df['text_length'].hist(bins=50, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Text Length Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=df['text_length'].median(), color='red', linestyle='--',
                label=f"Median: {df['text_length'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Train/Validation/Test split (80% / 10% / 10%)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_id']
)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, random_state=42, stratify=test_df['label_id']
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train labels: {dict(train_df['label_id'].value_counts())}")
print(f"Val labels:   {dict(val_df['label_id'].value_counts())}")
print(f"Test labels:  {dict(test_df['label_id'].value_counts())}")

---
## 6. 🎯 Zero-Shot Classification (No Training Needed!)

BART-Large-MNLI can classify text **without any training** using NLI.
It works by checking: *"Does this text entail the hypothesis 'This news is fake'?"*

In [ ]:
MODEL_NAME = 'facebook/bart-large-mnli'

# Create zero-shot classifier
print("📥 Loading BART-large-MNLI for zero-shot classification...")
zs_classifier = pipeline(
    'zero-shot-classification',
    model=MODEL_NAME,
    device=device
)
print("✅ Model loaded!")

In [ ]:
# Test on a few examples first
candidate_labels = ['fake news', 'real news']

print("🧪 Zero-Shot Classification Examples:")
print("=" * 60)

test_examples = test_df.sample(5, random_state=42)
for _, row in test_examples.iterrows():
    text = str(row['input_text'])[:1024]  # BART max ~1024 tokens
    true_label = 'FAKE' if row['label_id'] == 0 else 'REAL'
    
    result = zs_classifier(text, candidate_labels)
    pred_label = 'FAKE' if result['labels'][0] == 'fake news' else 'REAL'
    confidence = result['scores'][0]
    
    match = '✅' if pred_label == true_label else '❌'
    print(f"{match} True: {true_label:4s} | Pred: {pred_label:4s} | Conf: {confidence:.3f} | Text: {text[:80]}...")

print("=" * 60)

In [ ]:
# Evaluate zero-shot on the FULL test set
# (This takes a while because we process each example through the model)
print(f"🔄 Evaluating zero-shot on {len(test_df)} test samples...")
print("   (This may take 10-30 minutes depending on dataset size)")

zs_preds = []
zs_scores = []
batch_size = 1  # Process one at a time for zero-shot

for i, (_, row) in enumerate(test_df.iterrows()):
    text = str(row['input_text'])[:1024]
    try:
        result = zs_classifier(text, candidate_labels)
        pred = 0 if result['labels'][0] == 'fake news' else 1
        score = result['scores'][0]
    except Exception as e:
        pred = 1  # Default to real if error
        score = 0.5
    zs_preds.append(pred)
    zs_scores.append(score)
    
    if (i + 1) % 100 == 0:
        print(f"   Processed {i+1}/{len(test_df)}...")

# Calculate metrics
zs_acc = accuracy_score(test_df['label_id'].values, zs_preds)
zs_prec, zs_rec, zs_f1, _ = precision_recall_fscore_support(
    test_df['label_id'].values, zs_preds, average='weighted'
)

print(f"\n{'='*60}")
print(f"📊 ZERO-SHOT CLASSIFICATION RESULTS")
print(f"{'='*60}")
print(f"Accuracy:  {zs_acc:.4f}")
print(f"Precision: {zs_prec:.4f}")
print(f"Recall:    {zs_rec:.4f}")
print(f"F1-Score:  {zs_f1:.4f}")
print(f"{'='*60}")

# Classification report
print("\n📋 Classification Report (Zero-Shot):")
print(classification_report(test_df['label_id'].values, zs_preds,
                            target_names=['Fake', 'Real']))

# Save baseline metrics for comparison
baseline_zs_acc = zs_acc
baseline_zs_f1 = zs_f1

In [ ]:
# Confusion matrix for zero-shot
cm_zs = confusion_matrix(test_df['label_id'].values, zs_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_zs, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('Actual', fontsize=14)
ax.set_title('Confusion Matrix — Zero-Shot BART-MNLI', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Free zero-shot pipeline memory
del zs_classifier
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("🧹 Zero-shot model freed from memory.")

---
## 7. Load BART-MNLI for Fine-Tuning

Now we'll fine-tune BART-Large-MNLI for binary classification (Fake vs Real).
We adapt the 3-class NLI head (contradiction/neutral/entailment) to a 2-class head.

In [ ]:
from transformers import BartForSequenceClassification, BartTokenizer

MODEL_NAME = 'facebook/bart-large-mnli'

print("📥 Loading BART-large-MNLI for fine-tuning...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load with num_labels=2 for binary classification
# This will reinitialize the classification head (3-class NLI → 2-class)
model = BartForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True  # Needed because we change num_labels from 3 to 2
)
model = model.to(device)

# Print summary
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n{'='*60}")
print(f"MODEL SUMMARY")
print(f"{'='*60}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Model type: {model.config.model_type}")
print(f"Num labels: {model.config.num_labels}")
print(f"{'='*60}")

---
## 8. 🧊 Layer Freezing

BART-Large has 12 encoder + 12 decoder layers. We freeze early layers to:
- Preserve pre-trained language understanding
- Reduce memory usage and training time
- Prevent overfitting on smaller datasets

| Strategy | When to Use |
|----------|-------------|
| Freeze encoder + first 8 decoder layers | Small dataset (<5K) |
| Freeze encoder + first 6 decoder layers | Medium dataset (5K-20K) |
| **Freeze encoder only** | **Our case (20K+) ✅** |
| No freezing | Very large dataset (100K+) |

In [ ]:
# ============================================================
# 🔧 CONFIGURABLE: Freezing strategy
# ============================================================
FREEZE_ENCODER = True         # Freeze entire BART encoder
FREEZE_DECODER_LAYERS = 6     # Freeze first N decoder layers (of 12)

def freeze_bart_layers(model, freeze_encoder=True, freeze_decoder_layers=6):
    """Freeze BART layers for transfer learning."""
    
    # 1. Freeze shared embeddings
    if hasattr(model.model, 'shared'):
        for param in model.model.shared.parameters():
            param.requires_grad = False
    
    # 2. Freeze encoder
    if freeze_encoder:
        for param in model.model.encoder.parameters():
            param.requires_grad = False
    
    # 3. Freeze first N decoder layers
    for i in range(min(freeze_decoder_layers, len(model.model.decoder.layers))):
        for param in model.model.decoder.layers[i].parameters():
            param.requires_grad = False
    
    # 4. Classification head always trainable (already True)
    
    # Summary
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    
    print(f"\n{'='*60}")
    print(f"LAYER FREEZING SUMMARY")
    print(f"{'='*60}")
    print(f"Encoder:  {'🔴 Frozen' if freeze_encoder else '🟢 Trainable'}")
    print(f"Decoder:  Layers 0-{freeze_decoder_layers-1} 🔴 Frozen | Layers {freeze_decoder_layers}-11 🟢 Trainable")
    print(f"Classifier Head: 🟢 Trainable")
    print(f"{'─'*60}")
    print(f"Total parameters:     {total:>12,}")
    print(f"Frozen parameters:    {frozen:>12,} ({frozen/total*100:.1f}%)")
    print(f"Trainable parameters: {trainable:>12,} ({trainable/total*100:.1f}%)")
    print(f"{'='*60}")
    
    return model

model = freeze_bart_layers(model, FREEZE_ENCODER, FREEZE_DECODER_LAYERS)

---
## 9. Tokenize the Dataset

In [ ]:
MAX_LENGTH = 512  # BART max is 1024, but 512 is usually enough and saves memory

def tokenize_data(df, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a DataFrame into a HuggingFace Dataset."""
    encodings = tokenizer(
        df['input_text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': df['label_id'].values.tolist()
    })
    dataset.set_format('torch')
    return dataset

print("Tokenizing datasets...")
train_dataset = tokenize_data(train_df, tokenizer)
val_dataset = tokenize_data(val_df, tokenizer)
test_dataset = tokenize_data(test_df, tokenizer)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

---
## 10. ⚙️ Hyperparameter Configuration

In [ ]:
# ============================================================
# 🔧 HYPERPARAMETERS — Tune for best results
# ============================================================

HYPERPARAMS = {
    'learning_rate':        2e-5,
    'num_train_epochs':     5,
    'per_device_train_batch_size': 8,   # BART-large needs more memory, use smaller batch
    'per_device_eval_batch_size':  16,
    'weight_decay':         0.01,
    'warmup_ratio':         0.1,
    'gradient_accumulation_steps': 4,   # Effective batch = 8 * 4 = 32
    'max_grad_norm':        1.0,
    'lr_scheduler_type':    'cosine',
}

print("📋 Hyperparameter Configuration:")
print("─" * 50)
for k, v in HYPERPARAMS.items():
    print(f"  {k:<35} = {v}")
effective_batch = HYPERPARAMS['per_device_train_batch_size'] * HYPERPARAMS['gradient_accumulation_steps']
print(f"  {'effective_batch_size':<35} = {effective_batch}")
print("─" * 50)

---
## 11. Set Up Trainer

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

training_args = TrainingArguments(
    output_dir='./bart_fake_news_finetuned',
    num_train_epochs=HYPERPARAMS['num_train_epochs'],
    learning_rate=HYPERPARAMS['learning_rate'],
    per_device_train_batch_size=HYPERPARAMS['per_device_train_batch_size'],
    per_device_eval_batch_size=HYPERPARAMS['per_device_eval_batch_size'],
    weight_decay=HYPERPARAMS['weight_decay'],
    warmup_ratio=HYPERPARAMS['warmup_ratio'],
    gradient_accumulation_steps=HYPERPARAMS['gradient_accumulation_steps'],
    max_grad_norm=HYPERPARAMS['max_grad_norm'],
    lr_scheduler_type=HYPERPARAMS['lr_scheduler_type'],
    
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=3,
    
    logging_dir='./logs',
    logging_steps=50,
    report_to='none',
    
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("✅ Trainer configured with early stopping (patience=2)")

---
## 12. 🚀 Train the Model

In [ ]:
print("🚀 Starting fine-tuning BART-Large-MNLI...")
print(f"   Encoder: {'Frozen' if FREEZE_ENCODER else 'Trainable'}")
print(f"   Decoder: Layers 0-{FREEZE_DECODER_LAYERS-1} Frozen, {FREEZE_DECODER_LAYERS}-11 Trainable")
print(f"   Epochs: {HYPERPARAMS['num_train_epochs']} (with early stopping)")
print(f"   Learning rate: {HYPERPARAMS['learning_rate']}")
print(f"   Effective batch size: {HYPERPARAMS['per_device_train_batch_size'] * HYPERPARAMS['gradient_accumulation_steps']}")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")
print("=" * 60)

---
## 13. Plot Training History

In [ ]:
log_history = trainer.state.log_history

train_logs = [log for log in log_history if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in log_history if 'eval_loss' in log]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

if train_logs:
    steps = [log['step'] for log in train_logs]
    losses = [log['loss'] for log in train_logs]
    axes[0].plot(steps, losses, color='#e74c3c', linewidth=2)
    axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

if eval_logs:
    epochs = list(range(1, len(eval_logs) + 1))
    eval_losses = [log['eval_loss'] for log in eval_logs]
    eval_accs = [log.get('eval_accuracy', 0) for log in eval_logs]
    eval_f1s = [log.get('eval_f1', 0) for log in eval_logs]
    
    axes[1].plot(epochs, eval_losses, 'o-', color='#e67e22', linewidth=2, markersize=8)
    axes[1].set_title('Validation Loss', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs, eval_accs, 's-', color='#2ecc71', linewidth=2, markersize=8, label='Accuracy')
    axes[2].plot(epochs, eval_f1s, 'd-', color='#3498db', linewidth=2, markersize=8, label='F1-Score')
    axes[2].set_title('Validation Metrics', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].legend(fontsize=12)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0.5, 1.05])

plt.tight_layout()
plt.show()

---
## 14. 📊 Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("\n" + "=" * 60)
print("📊 FINAL TEST SET RESULTS (Fine-Tuned BART-MNLI)")
print("=" * 60)
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1-Score:  {test_results['eval_f1']:.4f}")
print(f"  Loss:      {test_results['eval_loss']:.4f}")
print("=" * 60)

# Compare with zero-shot baseline
print(f"\n🔄 COMPARISON (Zero-Shot vs Fine-Tuned):")
print(f"  Accuracy:  {baseline_zs_acc:.4f} → {test_results['eval_accuracy']:.4f} ({'+' if test_results['eval_accuracy'] > baseline_zs_acc else ''}{(test_results['eval_accuracy'] - baseline_zs_acc)*100:.2f}%)")
print(f"  F1-Score:  {baseline_zs_f1:.4f} → {test_results['eval_f1']:.4f} ({'+' if test_results['eval_f1'] > baseline_zs_f1 else ''}{(test_results['eval_f1'] - baseline_zs_f1)*100:.2f}%)")

In [ ]:
# Confusion Matrix & Classification Report
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("\n📋 Classification Report (Fine-Tuned):")
print(classification_report(true_labels, preds, target_names=['Fake', 'Real']))

cm = confusion_matrix(true_labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Zero-shot confusion matrix
sns.heatmap(cm_zs, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'],
            annot_kws={'size': 16})
axes[0].set_xlabel('Predicted', fontsize=14)
axes[0].set_ylabel('Actual', fontsize=14)
axes[0].set_title('Zero-Shot BART-MNLI', fontsize=16, fontweight='bold')

# Fine-tuned confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'],
            annot_kws={'size': 16})
axes[1].set_xlabel('Predicted', fontsize=14)
axes[1].set_ylabel('Actual', fontsize=14)
axes[1].set_title('Fine-Tuned BART-MNLI', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 15. 💾 Save the Fine-Tuned Model

In [ ]:
SAVE_DIR = './bart_fake_news_finetuned_final'

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✅ Model saved to: {SAVE_DIR}")
print(f"\nFiles saved:")
import os
for f in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024*1024)
    print(f"  {f} ({size_mb:.1f} MB)")

---
## 16. 🧪 Test with Custom Articles

In [ ]:
# Load the fine-tuned model
finetuned_clf = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=device,
    truncation=True
)

test_articles = [
    {
        'title': 'Fed Raises Interest Rates',
        'text': 'The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, '
                'bringing the benchmark rate to a range of 5.25% to 5.50%. Fed Chair Jerome Powell '
                'stated that the decision was made in response to persistent inflation.'
    },
    {
        'title': 'Miracle Cure Discovered',
        'text': 'SHOCKING: Scientists discover that drinking bleach cures all diseases! The government '
                'has been hiding this miracle cure for years. Share this before they delete it! '
                'Big pharma does not want you to know about this.'
    },
    {
        'title': 'Chocolate Extends Life',
        'text': 'A new study from an undisclosed university suggests eating chocolate every day can '
                'increase life expectancy by 10 years. Researchers tested on a small sample.'
    },
]

label_names = {0: '🔴 FAKE', 1: '🟢 REAL'}

print("\n" + "=" * 60)
print("🧪 PREDICTIONS ON CUSTOM ARTICLES")
print("=" * 60)
for article in test_articles:
    input_text = article['title'] + ' </s> ' + article['text']
    result = finetuned_clf(input_text)
    label_id = int(result[0]['label'].split('_')[-1])
    confidence = result[0]['score']
    
    print(f"\n📰 {article['title']}")
    print(f"   Prediction:  {label_names[label_id]}")
    print(f"   Confidence:  {confidence:.4f} ({confidence*100:.1f}%)")
    print(f"   Text:        {article['text'][:100]}...")
    print("─" * 60)

---
## 17. 📊 Final Summary & Model Comparison

In [ ]:
print("\n" + "=" * 60)
print("📊 FINAL MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"\n{'Method':<30} {'Accuracy':>10} {'F1-Score':>10}")
print("─" * 52)
print(f"{'Zero-Shot BART-MNLI':<30} {baseline_zs_acc:>10.4f} {baseline_zs_f1:>10.4f}")
print(f"{'Fine-Tuned BART-MNLI':<30} {test_results['eval_accuracy']:>10.4f} {test_results['eval_f1']:>10.4f}")
print("─" * 52)

improvement = (test_results['eval_f1'] - baseline_zs_f1) * 100
print(f"\n🎯 Fine-tuning improved F1 by {improvement:+.2f} percentage points")
print(f"\n💾 Fine-tuned model saved at: ./bart_fake_news_finetuned_final/")

---
## 📝 Tips for Better Results

| Tip | How |
|-----|-----|
| **Use a real dataset** | Ensure text is actual news articles, NOT synthetic/random |
| **Adjust freezing** | Try `FREEZE_DECODER_LAYERS = 4` or `8` |
| **Lower learning rate** | Try `1e-5` if model overfits |
| **Increase max length** | Use `MAX_LENGTH = 1024` for longer articles |
| **More epochs** | Set `num_train_epochs = 10` with early stopping patience=3 |
| **Data augmentation** | Synonym replacement, back-translation |
| **NLI-style fine-tuning** | Frame as entailment pairs instead of classification |